# 🔬 Augmentation CSV par moyenne de paires — Piste 2 (Dr. Sarun)
## Générer ~18 000 spectres CSV synthétiques (comme le volume .npy)
### CMKL University · Stage 2026

---

**Consigne Dr. Sarun** : *"Créer plus de spectres CSV (pour avoir 18000 au
mieux) — prendre deux spectres du même type et faire la moyenne."*

**Méthode** : pour chaque classe, générer des spectres synthétiques en
moyennant deux spectres bruités **de la même classe**, tirés aléatoirement
(avec remise entre générations), jusqu'à atteindre **600 spectres par classe**
— exactement le volume du dataset `.npy` (30 × 600 = 18 000).

**⚠️ Limite méthodologique à garder en tête** :
- Une moyenne de 2 spectres bruités a tendance à **réduire le bruit
  indépendant** entre eux — les spectres synthétiques seront donc légèrement
  plus "propres" que de vraies nouvelles mesures, pas une nouvelle
  information indépendante.
- Pour les classes avec très peu de spectres de base (ex: PP, POM), le nombre
  de paires *vraiment* distinctes est limité — on tire avec remise, donc
  certaines paires reviendront plusieurs fois avec des combinaisons proches.

**Sécurité méthodologique** : l'augmentation ne touche **QUE** le Train CSV.
Val et Test restent strictement les originaux, jamais synthétiques — sinon on
biaiserait l'évaluation.

**Ce notebook réutilise `ssl_backbone_mixed.pth`** (Phase 1 déjà faite) —
seules Phase 2 et 3 sont relancées, sur .npy complet + CSV augmenté à 18000.


---
## ⚙️ Section 0 — Imports & Configuration


In [ ]:
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
for pkg in ['scikit-learn', 'seaborn']:
    try: __import__(pkg.replace('-','_'))
    except ImportError: install(pkg)
print('✓ Packages prêts')

In [ ]:
import os, glob, re, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HOME   = os.path.expanduser('~')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

In [ ]:
from torch.utils.tensorboard import SummaryWriter

LOG_DIR = os.path.join(HOME, 'runs', 'patchtst_csv_augmented')
os.makedirs(LOG_DIR, exist_ok=True)
writer  = SummaryWriter(LOG_DIR)
print(f'✓ TensorBoard logs → {LOG_DIR}')

def safe_log(writer, *args, method='add_scalar', **kwargs):
    try: getattr(writer, method)(*args, **kwargs)
    except Exception: pass

In [ ]:
CFG = {
    'L'         : 6700,
    'WN_MIN'    : 650,
    'WN_MAX'    : 4000,
    'WN_STEP'   : 0.5,

    'patch_size' : 320,
    'stride'     : 256,

    'd_model'   : 256,
    'n_heads'   : 16,
    'n_layers'  : 5,
    'd_ff'      : 512,
    'dropout'   : 0.1,

    'alpha'        : 1.0,
    'beta'         : 0.5,
    'probe_epochs' : 80,
    'probe_lr'     : 1e-3,
    'ft_epochs'    : 80,
    'ft_lr'        : 1e-5,
    'batch_size'   : 32,

    'target_per_class' : 600,   # même volume que .npy (30×600=18000)

    'ssl_path'   : os.path.join(HOME, 'models', 'ssl_backbone_mixed.pth'),   # RÉUTILISÉ
    'probe_path' : os.path.join(HOME, 'models', 'probe_model_csvaugmented.pth'),
    'final_path' : os.path.join(HOME, 'models', 'final_model_csvaugmented.pth'),

    'noise_variant' : 'Upto30SNR',
}
os.makedirs(os.path.join(HOME, 'models'), exist_ok=True)

WN_GRID   = np.arange(CFG['WN_MIN'], CFG['WN_MAX'], CFG['WN_STEP'])
CFG['L']  = len(WN_GRID)
L = CFG['L']
N_PATCHES = (L - CFG['patch_size']) // CFG['stride'] + 2
print(f'L = {L}, N_PATCHES = {N_PATCHES}')

In [ ]:
ASSUMED_CLASSES = [
    'ABS', 'ACRYLIC', 'CELLULOSE', 'CHITOSAN', 'ENR', 'EPDM', 'EVA', 'HDPE',
    'LDPE', 'NYLON', 'PBAT', 'PBS', 'PC', 'PEEK', 'PEI', 'PET',
    'PF THERMOPLASTIC', 'PF THERMOSET', 'PHB', 'PLA', 'PMMA', 'POM', 'PP',
    'PS', 'PTFE', 'PU', 'PVA', 'PVC', 'PVDF', 'SAN',
]
N_CLASSES = len(ASSUMED_CLASSES)
CFG['N_CLASSES'] = N_CLASSES
le = LabelEncoder()
le.fit(ASSUMED_CLASSES)
print(f'{N_CLASSES} classes')

---
## 📊 Section 1 — Données .npy (complètes, inchangées)


In [ ]:
NPY_ROOT  = os.path.join(HOME, 'data', '2026-FTIR-Preprocesed')
TRAIN_DIR = os.path.join(NPY_ROOT, '1.1 TrainingSet - UptoY dB')
TEST_DIR  = os.path.join(NPY_ROOT, '1.2 TestSet - UptoY dB')

def npy_path(base_dir, filename):
    p = os.path.join(base_dir, filename)
    if not os.path.exists(p): print(f'  ✗ INTROUVABLE : {p}')
    return p

noise = CFG['noise_variant']
npy_train_clean = np.load(npy_path(TRAIN_DIR, 'TrainGroundTruthSet_Pre.npy'))
npy_train_noisy = np.load(npy_path(TRAIN_DIR, f'TrainNoisySet_{noise}_Pre.npy'))
npy_test_clean  = np.load(npy_path(TEST_DIR,  'TestGroundTruthSet_Pre.npy'))
npy_test_noisy  = np.load(npy_path(TEST_DIR,  f'TestNoisySet_{noise}_Pre.npy'))

assert npy_train_clean.shape[1] == L
N_PER_CLASS_NPY_TRAIN = npy_train_clean.shape[0] // N_CLASSES
N_PER_CLASS_NPY_TEST  = npy_test_clean.shape[0]  // N_CLASSES
npy_labels_train_full = np.repeat(np.arange(N_CLASSES), N_PER_CLASS_NPY_TRAIN)
npy_labels_test       = np.repeat(np.arange(N_CLASSES), N_PER_CLASS_NPY_TEST)

N_VAL_PER_CLASS_NPY = 30
val_idx_npy, train_idx_npy = [], []
for c in range(N_CLASSES):
    cls_idx = np.where(npy_labels_train_full == c)[0]
    rng = np.random.RandomState(SEED)
    rng.shuffle(cls_idx)
    val_idx_npy.extend(cls_idx[:N_VAL_PER_CLASS_NPY])
    train_idx_npy.extend(cls_idx[N_VAL_PER_CLASS_NPY:])
val_idx_npy = np.array(val_idx_npy); train_idx_npy = np.array(train_idx_npy)

npy_train_noisy_eff = npy_train_noisy[train_idx_npy]
npy_train_clean_eff = npy_train_clean[train_idx_npy]
npy_labels_train_eff = npy_labels_train_full[train_idx_npy]

npy_val_noisy = npy_train_noisy[val_idx_npy]
npy_val_clean = npy_train_clean[val_idx_npy]
npy_labels_val = npy_labels_train_full[val_idx_npy]

print(f'✓ .npy Train : {len(npy_train_noisy_eff)}   Val : {len(npy_val_noisy)}   Test : {len(npy_test_noisy)}')

---
## 📁 Section 2 — Chargement CSV (Val/Test FIXES, comme d'habitude)


In [ ]:
CSV_ROOT = os.path.join(HOME, 'data', '2026-FirstDataSet', '2026 - Complete FTIR Dataset')
PATHS_CSV = {
    '2023_base' : os.path.join(CSV_ROOT, '2023 Dataset - 22 MP Types with 10 Clean and 60 Noisy'),
    '2025_ext'  : os.path.join(CSV_ROOT, '2025 Dataset 1 - Same 22 MP Types - Add 40 Spectra'),
    '2025_new'  : os.path.join(CSV_ROOT, '2025 Dataset 2 - New 9 MP Types - 50 Clean and 100 Noisy'),
}
EXCLUDE_FILES = {'ref.csv', 'reference.csv', 'background.csv', 'bg.csv'}

def is_noisy_csv(filepath):
    p = str(filepath).lower()
    if any(k in p for k in ['noisy', '_sd', '-sd', 'sd_']): return True
    if any(k in p for k in ['clean', '_rm', '-rm', 'rm_']): return False
    return False

def extract_label_csv(filepath):
    name = Path(filepath).stem.upper()
    for pattern in ['_SD_', '_RM_', '_NOISY', '_CLEAN', 'PARTICLE', '-NOISY',
                    '-CLEAN', '_50', '_60', '_40', '_100', '_10', '_30',
                    ' SPECTRUMS', ' SPECTUMS', 'ADD_40', '-ADD_40']:
        name = name.replace(pattern, ' ')
    name = re.sub(r'\d+', '', name)
    name = re.sub(r'\bNEW\b|\bJAN\b|\bX\b', '', name)
    name = ' '.join(name.replace('_', ' ').replace('-', ' ').split())
    MAPPING = {
        'NYLON PARTICLE' : 'NYLON', 'PTEE' : 'PTFE', 'PTFE' : 'PTFE',
        'PF THERMOPLASTIC CLEAN' : 'PF THERMOPLASTIC',
        'PF THERMOSET CLEAN'     : 'PF THERMOSET',
    }
    if name in MAPPING: return MAPPING[name]
    if name in ASSUMED_CLASSES: return name
    for cls in ASSUMED_CLASSES:
        if cls in name or name in cls: return cls
    return None

def read_csv_multispectra(filepath, sep=','):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        start_idx = 0
        for i, line in enumerate(lines):
            parts = line.strip().split(sep)
            if len(parts) >= 2:
                try:
                    float(parts[0].replace(',', '.'))
                    start_idx = i; break
                except ValueError: continue
        valid = ''.join(lines[start_idx:])
        headers = lines[start_idx-1].strip().split(sep) if start_idx > 0 else []
        try:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal=',')
        except Exception:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal='.')
        cols = []
        for ci, cn in enumerate(df.columns):
            h = headers[ci].upper() if ci < len(headers) else ''
            v = str(df[cn].iloc[0]).upper()
            if any(k in h for k in ['AIR','BACKGROUND','BG']): continue
            if any(k in v for k in ['AIR','BACKGROUND','BG']): continue
            cols.append(cn)
        df = df[cols].apply(pd.to_numeric, errors='coerce')
        df = df.dropna(subset=[df.columns[0]])
        if len(df) < 100: return None
        wn    = df.iloc[:, 0].values.astype(float)
        order = np.argsort(wn); wn = wn[order]
        spectra = []
        for c in range(1, df.shape[1]):
            ab = df.iloc[order, c].values.astype(float)
            if np.isnan(ab).all() or ab.std() < 1e-10: continue
            nans = np.isnan(ab)
            if nans.any():
                ab[nans] = np.interp(np.where(nans)[0], np.where(~nans)[0], ab[~nans])
            spectra.append(ab.astype(np.float32))
        return (wn, spectra) if spectra else None
    except Exception: return None

print('✓ Fonctions de lecture définies')

In [ ]:
print('Chargement des CSV bruités...')
csv_records = []
for src_name, folder in PATHS_CSV.items():
    if not os.path.exists(folder): continue
    files = glob.glob(os.path.join(folder, '**/*.csv'), recursive=True)
    for fp in files:
        if Path(fp).name.lower() in EXCLUDE_FILES: continue
        label = extract_label_csv(fp)
        if label is None: continue
        result = read_csv_multispectra(fp)
        if result is None: continue
        wn, spectra_list = result
        if not is_noisy_csv(fp): continue
        for sp in spectra_list:
            sp_interp = np.interp(WN_GRID, wn, sp).astype(np.float32)
            csv_records.append({'label': label, 'spectrum': sp_interp})

df_csv_noisy = pd.DataFrame(csv_records)
df_csv_noisy['label_enc'] = le.transform(df_csv_noisy['label'])
print(f'Total CSV bruités : {len(df_csv_noisy)}')

# ── Split fixe Val/Test — IDENTIQUE à tous les autres notebooks ───────────
csv_noisy_counts = Counter(df_csv_noisy['label_enc'])
csv_singleton = {k for k, v in csv_noisy_counts.items() if v < 3}
df_csv_multi  = df_csv_noisy[~df_csv_noisy['label_enc'].isin(csv_singleton)]
df_csv_single = df_csv_noisy[ df_csv_noisy['label_enc'].isin(csv_singleton)]

idx_tr_csv, idx_valtest_csv = train_test_split(
    range(len(df_csv_multi)), test_size=0.3, random_state=SEED, stratify=df_csv_multi['label_enc'])
idx_val_csv, idx_test_csv = train_test_split(idx_valtest_csv, test_size=0.5, random_state=SEED)

df_csv_train_original = pd.concat([df_csv_multi.iloc[idx_tr_csv], df_csv_single]).reset_index(drop=True)
df_csv_val  = df_csv_multi.iloc[idx_val_csv].reset_index(drop=True)
df_csv_test = df_csv_multi.iloc[idx_test_csv].reset_index(drop=True)

print(f'CSV Train original (avant augmentation) : {len(df_csv_train_original)}')
print(f'CSV Val  (FIXE, jamais augmenté) : {len(df_csv_val)}')
print(f'CSV Test (FIXE, jamais augmenté) : {len(df_csv_test)}')

---
## 🧪 Section 3 — Augmentation par moyenne de paires (SEULEMENT sur le Train)


In [ ]:
# ── Distribution AVANT augmentation ────────────────────────────────────────
counts_before = df_csv_train_original['label_enc'].value_counts().sort_index()
print('Distribution CSV Train AVANT augmentation :')
for cls_idx in range(N_CLASSES):
    n = counts_before.get(cls_idx, 0)
    bar = '█' * (n // 5)
    print(f'  {le.classes_[cls_idx]:20s} : {n:4d} {bar}')

In [ ]:
def generate_synthetic_by_averaging(class_spectra, n_synthetic, seed):
    """
    Génère n_synthetic spectres en moyennant des paires aléatoires
    (avec remise entre tirages) de spectres de la MÊME classe.
    """
    rng = np.random.RandomState(seed)
    n_available = len(class_spectra)
    synthetic = []

    for _ in range(n_synthetic):
        if n_available >= 2:
            i, j = rng.choice(n_available, size=2, replace=False)
        else:
            i = j = 0   # cas limite : une seule mesure dispo pour cette classe
        avg = (class_spectra[i] + class_spectra[j]) / 2.0
        synthetic.append(avg.astype(np.float32))

    return np.array(synthetic)


TARGET = CFG['target_per_class']
augmented_records = []

print(f'Génération de spectres synthétiques (cible : {TARGET}/classe)...')
for cls_idx in range(N_CLASSES):
    cls_spectra = np.stack(
        df_csv_train_original[df_csv_train_original['label_enc'] == cls_idx]['spectrum'].values
    ) if counts_before.get(cls_idx, 0) > 0 else np.zeros((0, L), dtype=np.float32)

    n_existing = len(cls_spectra)
    n_needed   = max(0, TARGET - n_existing)

    # Garder les originaux
    for sp in cls_spectra:
        augmented_records.append({'label_enc': cls_idx, 'spectrum': sp, 'is_synthetic': False})

    # Ajouter les synthétiques si besoin
    if n_needed > 0 and n_existing > 0:
        synth = generate_synthetic_by_averaging(cls_spectra, n_needed, seed=SEED + cls_idx)
        for sp in synth:
            augmented_records.append({'label_enc': cls_idx, 'spectrum': sp, 'is_synthetic': True})

    print(f'  {le.classes_[cls_idx]:20s} : {n_existing:4d} originaux + {max(0,n_needed):4d} synthétiques = {n_existing+max(0,n_needed):4d}')

df_csv_train_augmented = pd.DataFrame(augmented_records)
print(f'\n✓ Total CSV Train augmenté : {len(df_csv_train_augmented)} spectres')
print(f'  dont synthétiques : {df_csv_train_augmented["is_synthetic"].sum()}')

In [ ]:
# ── Vérification visuelle : spectre original vs synthétique (même classe) ──
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
sample_classes = [0, 10, 20, 22]   # POM et PP inclus (classes rares)

for ax, cls_idx in zip(axes.flatten(), sample_classes):
    orig = df_csv_train_augmented[(df_csv_train_augmented['label_enc']==cls_idx) &
                                    (~df_csv_train_augmented['is_synthetic'])]
    synth = df_csv_train_augmented[(df_csv_train_augmented['label_enc']==cls_idx) &
                                     (df_csv_train_augmented['is_synthetic'])]
    if len(orig) > 0:
        ax.plot(WN_GRID, orig.iloc[0]['spectrum'], color='steelblue', lw=1.0,
                alpha=0.8, label='Original')
    if len(synth) > 0:
        ax.plot(WN_GRID, synth.iloc[0]['spectrum'], color='coral', lw=1.0,
                alpha=0.8, label='Synthétique (moyenne)')
    ax.set_title(le.classes_[cls_idx], fontweight='bold')
    ax.invert_xaxis(); ax.legend(fontsize=8)

plt.suptitle('Vérification qualitative — Original vs Synthétique (même classe)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🎯 Section 4 — Cibles de denoising (moyenne de classe, comme d'habitude)


In [ ]:
# ── Recharger les spectres propres pour la référence de denoising ─────────
csv_clean_records = []
for src_name, folder in PATHS_CSV.items():
    if not os.path.exists(folder): continue
    files = glob.glob(os.path.join(folder, '**/*.csv'), recursive=True)
    for fp in files:
        if Path(fp).name.lower() in EXCLUDE_FILES: continue
        label = extract_label_csv(fp)
        if label is None: continue
        result = read_csv_multispectra(fp)
        if result is None: continue
        wn, spectra_list = result
        if is_noisy_csv(fp): continue   # on veut les PROPRES ici
        for sp in spectra_list:
            sp_interp = np.interp(WN_GRID, wn, sp).astype(np.float32)
            csv_clean_records.append({'label': label, 'spectrum': sp_interp})

df_csv_clean = pd.DataFrame(csv_clean_records)
df_csv_clean['label_enc'] = le.transform(df_csv_clean['label'])

csv_clean_reference = {}
for cls_idx in range(N_CLASSES):
    subset = df_csv_clean[df_csv_clean['label_enc'] == cls_idx]['spectrum']
    if len(subset) > 0:
        csv_clean_reference[cls_idx] = np.mean(np.stack(subset.values), axis=0).astype(np.float32)
csv_global_clean_mean = (np.mean(np.stack(df_csv_clean['spectrum'].values), axis=0).astype(np.float32)
                         if len(df_csv_clean) > 0 else np.zeros(L, dtype=np.float32))
print(f'Référence propre disponible pour {len(csv_clean_reference)}/{N_CLASSES} classes')

---
## 🏗️ Section 5 — Datasets & Architecture


In [ ]:
class MixedDataset(Dataset):
    def __init__(self, npy_noisy, npy_clean, npy_labels,
                 csv_df, csv_clean_ref, csv_global_mean):
        self.npy_noisy  = npy_noisy.astype(np.float32)
        self.npy_clean  = npy_clean.astype(np.float32)
        self.npy_labels = npy_labels.astype(np.int64)
        self.n_npy = len(npy_labels)
        self.csv_spectra = np.stack(csv_df['spectrum'].values).astype(np.float32)
        self.csv_labels  = csv_df['label_enc'].values.astype(np.int64)
        self.csv_clean_ref   = csv_clean_ref
        self.csv_global_mean = csv_global_mean
        self.n_csv = len(self.csv_labels)
    def __len__(self): return self.n_npy + self.n_csv
    def __getitem__(self, idx):
        if idx < self.n_npy:
            x       = torch.tensor(self.npy_noisy[idx], dtype=torch.float32)
            x_clean = torch.tensor(self.npy_clean[idx], dtype=torch.float32)
            y       = torch.tensor(self.npy_labels[idx], dtype=torch.long)
        else:
            i = idx - self.n_npy
            x = torch.tensor(self.csv_spectra[i], dtype=torch.float32)
            y = torch.tensor(self.csv_labels[i], dtype=torch.long)
            ref = self.csv_clean_ref.get(int(self.csv_labels[i]), self.csv_global_mean)
            x_clean = torch.tensor(ref, dtype=torch.float32)
        mu, sigma = x.mean(), x.std() + 1e-8
        x       = (x - mu) / sigma
        x_clean = (x_clean - mu) / sigma
        return x, x_clean, y

class SimpleMultiTaskDataset(Dataset):
    def __init__(self, noisy, clean, labels):
        self.noisy  = noisy.astype(np.float32)
        self.clean  = clean.astype(np.float32)
        self.labels = labels.astype(np.int64)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        x       = torch.tensor(self.noisy[idx], dtype=torch.float32)
        x_clean = torch.tensor(self.clean[idx], dtype=torch.float32)
        y       = torch.tensor(self.labels[idx], dtype=torch.long)
        mu, sigma = x.mean(), x.std() + 1e-8
        x       = (x - mu) / sigma
        x_clean = (x_clean - mu) / sigma
        return x, x_clean, y

# ── Train : .npy complet + CSV AUGMENTÉ (18000) ────────────────────────────
train_dataset = MixedDataset(npy_train_noisy_eff, npy_train_clean_eff, npy_labels_train_eff,
                              df_csv_train_augmented, csv_clean_reference, csv_global_clean_mean)

val_npy_dataset = SimpleMultiTaskDataset(npy_val_noisy, npy_val_clean, npy_labels_val)

csv_val_clean_targets = np.stack([csv_clean_reference.get(int(l), csv_global_clean_mean)
                                  for l in df_csv_val['label_enc'].values])
val_csv_dataset = SimpleMultiTaskDataset(np.stack(df_csv_val['spectrum'].values),
                                         csv_val_clean_targets, df_csv_val['label_enc'].values)

test_npy_dataset = SimpleMultiTaskDataset(npy_test_noisy, npy_test_clean, npy_labels_test)

csv_test_clean_targets = np.stack([csv_clean_reference.get(int(l), csv_global_clean_mean)
                                   for l in df_csv_test['label_enc'].values])
test_csv_dataset = SimpleMultiTaskDataset(np.stack(df_csv_test['spectrum'].values),
                                          csv_test_clean_targets, df_csv_test['label_enc'].values)

print(f'✓ train_dataset : {len(train_dataset)} (.npy={train_dataset.n_npy}, CSV augmenté={train_dataset.n_csv})')
print(f'✓ val_npy : {len(val_npy_dataset)}   val_csv : {len(val_csv_dataset)}')
print(f'✓ test_npy : {len(test_npy_dataset)}   test_csv : {len(test_csv_dataset)} (FIXE, jamais augmenté)')

In [ ]:
weight_npy = 1.0 / train_dataset.n_npy
weight_csv = 1.0 / train_dataset.n_csv
sample_weights = np.concatenate([
    np.full(train_dataset.n_npy, weight_npy),
    np.full(train_dataset.n_csv, weight_csv),
])
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float32),
    num_samples=len(train_dataset), replacement=True)

train_loader    = DataLoader(train_dataset, batch_size=CFG['batch_size'], sampler=sampler, num_workers=0)
val_npy_loader  = DataLoader(val_npy_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
val_csv_loader  = DataLoader(val_csv_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
test_npy_loader = DataLoader(test_npy_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
test_csv_loader = DataLoader(test_csv_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
print('✓ DataLoaders prêts (sampler 50/50 npy/CSV, comme le modèle mixte de référence)')

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, L, patch_size, stride, d_model):
        super().__init__()
        self.P, self.S, self.D = patch_size, stride, d_model
        self.N = (L - patch_size) // stride + 2
        self.patch_proj = nn.Linear(patch_size, d_model)
        self.pos_embed  = nn.Embedding(self.N, d_model)
        self.dropout    = nn.Dropout(0.1)
    def get_raw_patches(self, x):
        B = x.shape[0]
        pad = x[:, -1:].expand(B, self.S)
        x_pad = torch.cat([x, pad], dim=1)
        return x_pad.unfold(1, self.P, self.S)
    def forward(self, x):
        patches  = self.get_raw_patches(x)
        content  = self.patch_proj(patches)
        pos_vecs = self.pos_embed(torch.arange(self.N, device=x.device))
        return self.dropout(content + pos_vecs)

class ConformerFFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.W1, self.V, self.W2 = (nn.Linear(d_model, d_ff), nn.Linear(d_model, d_ff),
                                     nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = self.norm(x)
        return self.drop(self.W2(F.silu(self.W1(x)) * self.V(x)))

class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.pw1  = nn.Conv1d(d_model, 2*d_model, 1)
        self.glu  = nn.GLU(dim=1)
        self.dw   = nn.Conv1d(d_model, d_model, kernel_size, padding=kernel_size//2, groups=d_model)
        self.bn   = nn.BatchNorm1d(d_model)
        self.act  = nn.SiLU()
        self.pw2  = nn.Conv1d(d_model, d_model, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        r = x
        x = self.norm(x).transpose(1,2)
        x = self.glu(self.pw1(x))
        x = self.act(self.bn(self.dw(x)))
        x = self.drop(self.pw2(x)).transpose(1,2)
        return r + x

class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, kernel_size=31):
        super().__init__()
        self.ffn1 = ConformerFFN(d_model, d_ff, dropout)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.conv = ConformerConvModule(d_model, kernel_size, dropout)
        self.ffn2 = ConformerFFN(d_model, d_ff, dropout)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = x + 0.5 * self.ffn1(x)
        xn = self.attn_norm(x)
        x  = x + self.drop(self.attn(xn, xn, xn)[0])
        x  = self.conv(x)
        x  = x + 0.5 * self.ffn2(x)
        return self.norm(x)

class TransformerBackbone(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            ConformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        for l in self.layers: x = l(x)
        return self.norm(x)

class ClassificationHead(nn.Module):
    def __init__(self, d_model, n_classes, dropout=0.1, hidden_dim=None):
        super().__init__()
        if hidden_dim is None: hidden_dim = d_model // 2
        self.attn_pool = nn.Linear(d_model, 1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, n_classes))
    def forward(self, z):
        w = F.softmax(self.attn_pool(z), dim=1)
        return self.head((w * z).sum(dim=1))

class DenoisingHead(nn.Module):
    def __init__(self, d_model, patch_size, n_patches, stride, spectrum_length):
        super().__init__()
        self.P, self.S, self.N, self.L = patch_size, stride, n_patches, spectrum_length
        self.proj = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
                                   nn.GELU(), nn.Linear(d_model, patch_size))
    def forward(self, z):
        B = z.shape[0]
        pr = self.proj(z)
        out = torch.zeros(B, self.L+self.S, device=z.device)
        cnt = torch.zeros(self.L+self.S, device=z.device)
        for k in range(self.N):
            s = k * self.S
            out[:, s:s+self.P] += pr[:, k, :]
            cnt[s:s+self.P]    += 1
        return (out / cnt.clamp(min=1))[:, :self.L]

class PatchTSTMultiTask(nn.Module):
    def __init__(self, patch_embed, backbone, class_head, denoise_head, alpha=1.0, beta=0.5):
        super().__init__()
        self.patch_embed, self.backbone = patch_embed, backbone
        self.class_head, self.denoise_head = class_head, denoise_head
        self.alpha, self.beta = alpha, beta
    def encode(self, x):
        return self.backbone(self.patch_embed(x))
    def classify(self, x):
        return self.class_head(self.encode(x))
    def forward(self, x, y=None, clean_target=None):
        z = self.encode(x)
        logits   = self.class_head(z)
        denoised = self.denoise_head(z)
        loss = None
        if y is not None and clean_target is not None:
            loss_clf = F.cross_entropy(logits, y, label_smoothing=0.1)
            loss_den = F.mse_loss(denoised, clean_target)
            loss = self.alpha * loss_clf + self.beta * loss_den
        return logits, denoised, loss

print('✓ Architecture définie')

---
## 📦 Section 6 — Charger le backbone SSL MIXTE (réutilisé, pas de nouveau Phase 1)


In [ ]:
patch_embed  = PatchEmbedding(CFG['L'], CFG['patch_size'], CFG['stride'], CFG['d_model']).to(DEVICE)
backbone     = TransformerBackbone(CFG['d_model'], CFG['n_heads'], CFG['n_layers'],
                                    CFG['d_ff'], CFG['dropout']).to(DEVICE)

ckpt = torch.load(CFG['ssl_path'], map_location=DEVICE, weights_only=False)
patch_embed.load_state_dict(ckpt['patch_embed'])
backbone.load_state_dict(ckpt['backbone'])
print(f'✓ Backbone SSL mixte chargé (val MSE = {ckpt["ssl_val_loss"]:.6f})')

class_head   = ClassificationHead(CFG['d_model'], CFG['N_CLASSES'], dropout=0.1).to(DEVICE)
denoise_head = DenoisingHead(CFG['d_model'], CFG['patch_size'], N_PATCHES,
                              CFG['stride'], CFG['L']).to(DEVICE)

total = sum(p.numel() for m in [patch_embed, backbone, class_head, denoise_head]
            for p in m.parameters() if p.requires_grad)
print(f'Total paramètres : {total:,}')

---
## 🛠️ Section 7 — Fonctions d'entraînement


In [ ]:
def snr_db(clean, signal):
    noise_power  = ((signal - clean) ** 2).mean(dim=-1) + 1e-8
    signal_power = (clean ** 2).mean(dim=-1) + 1e-8
    return 10 * torch.log10(signal_power / noise_power)

def multitask_train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, x_clean, y in loader:
        x, x_clean, y = x.to(DEVICE), x_clean.to(DEVICE), y.to(DEVICE)
        logits, denoised, loss = model(x, y=y, clean_target=x_clean)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item(); n += len(y)
    return total_loss/n, correct/n

@torch.no_grad()
def multitask_eval_epoch(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, x_clean, y in loader:
        x, x_clean, y = x.to(DEVICE), x_clean.to(DEVICE), y.to(DEVICE)
        logits, denoised, loss = model(x, y=y, clean_target=x_clean)
        total_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item(); n += len(y)
    return total_loss/n, correct/n

print('✓ Fonctions définies')

---
## 🔍 Phase 2 — Linear Probing (train mixte, .npy complet + CSV augmenté)


In [ ]:
for p in patch_embed.parameters(): p.requires_grad = False
for p in backbone.parameters():    p.requires_grad = False

probe_model = PatchTSTMultiTask(patch_embed, backbone, class_head, denoise_head,
                                 alpha=CFG['alpha'], beta=CFG['beta']).to(DEVICE)
probe_optimizer = AdamW(filter(lambda p: p.requires_grad, probe_model.parameters()),
                        lr=CFG['probe_lr'], weight_decay=1e-4)

best_probe_acc = 0.0
print(f'=== Phase 2 : Linear Probing ({CFG["probe_epochs"]} époques) ===')
for epoch in range(1, CFG['probe_epochs']+1):
    tr_loss, tr_acc = multitask_train_epoch(probe_model, train_loader, probe_optimizer)
    _, va_npy_acc = multitask_eval_epoch(probe_model, val_npy_loader)
    _, va_csv_acc = multitask_eval_epoch(probe_model, val_csv_loader)

    safe_log(writer, 'Phase2_Probe/Acc_npy', {'Train': tr_acc, 'Val': va_npy_acc}, epoch, method='add_scalars')
    safe_log(writer, 'Phase2_Probe/Acc_csv', {'Val': va_csv_acc}, epoch, method='add_scalars')

    avg_acc = (va_npy_acc + va_csv_acc) / 2
    if avg_acc > best_probe_acc:
        best_probe_acc = avg_acc
        torch.save(probe_model.state_dict(), CFG['probe_path'])

    if epoch % 20 == 0 or epoch == 1:
        print(f'  epoch {epoch:3d} : train={tr_acc:.2%}  val_npy={va_npy_acc:.2%}  val_csv={va_csv_acc:.2%}')

print(f'\n✓ Meilleure moyenne : {best_probe_acc:.2%}')

---
## 🎯 Phase 3 — Fine-tuning complet


In [ ]:
probe_model.load_state_dict(torch.load(CFG['probe_path'], map_location=DEVICE, weights_only=False))
for p in probe_model.parameters(): p.requires_grad = True

ft_optimizer = AdamW([
    {'params': probe_model.patch_embed.parameters(),  'lr': CFG['ft_lr']},
    {'params': probe_model.backbone.parameters(),     'lr': CFG['ft_lr']},
    {'params': probe_model.class_head.parameters(),   'lr': CFG['ft_lr']*10},
    {'params': probe_model.denoise_head.parameters(), 'lr': CFG['ft_lr']*10},
], weight_decay=1e-4)
ft_scheduler = CosineAnnealingLR(ft_optimizer, T_max=CFG['ft_epochs'], eta_min=1e-6)

best_ft_acc = 0.0
print(f'=== Phase 3 : Fine-tuning ({CFG["ft_epochs"]} époques) ===')
for epoch in range(1, CFG['ft_epochs']+1):
    tr_loss, tr_acc = multitask_train_epoch(probe_model, train_loader, ft_optimizer)
    _, va_npy_acc = multitask_eval_epoch(probe_model, val_npy_loader)
    _, va_csv_acc = multitask_eval_epoch(probe_model, val_csv_loader)
    ft_scheduler.step()

    safe_log(writer, 'Phase3_FT/Acc_npy', {'Train': tr_acc, 'Val': va_npy_acc}, epoch, method='add_scalars')
    safe_log(writer, 'Phase3_FT/Acc_csv', {'Val': va_csv_acc}, epoch, method='add_scalars')

    if epoch % 5 == 0:
        torch.save({'model_state': probe_model.state_dict(), 'epoch': epoch, 'cfg': CFG},
                   os.path.join(HOME, 'models', 'checkpoint_csvaugmented_latest.pth'))

    avg_acc = (va_npy_acc + va_csv_acc) / 2
    if avg_acc > best_ft_acc:
        best_ft_acc = avg_acc
        torch.save({'model_state': probe_model.state_dict(), 'cfg': CFG,
                    'le_classes': le.classes_, 'val_acc': best_ft_acc}, CFG['final_path'])

    if epoch % 10 == 0 or epoch == 1:
        print(f'  epoch {epoch:3d} : train={tr_acc:.2%}  val_npy={va_npy_acc:.2%}  val_csv={va_csv_acc:.2%}')

writer.flush()
print(f'\n✓ Meilleure moyenne : {best_ft_acc:.2%}')
print(f'✓ Sauvegardé → {CFG["final_path"]}')
print('🔴 TÉLÉCHARGE ce fichier vers ton Mac')

---
## 📊 Évaluation finale — comparaison avec le modèle mixte de référence (CSV non augmenté)


In [ ]:
ckpt = torch.load(CFG['final_path'], map_location=DEVICE, weights_only=False)
probe_model.load_state_dict(ckpt['model_state'])
probe_model.eval()

def evaluate_full(model, loader):
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, x_clean, y in loader:
            logits, _, _ = model(x.to(DEVICE))
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y.numpy())
    return accuracy_score(all_labels, all_preds)

acc_npy = evaluate_full(probe_model, test_npy_loader)
acc_csv = evaluate_full(probe_model, test_csv_loader)

print('═'*70)
print('  RÉSULTAT — CSV augmenté à 18000 (600/classe) par moyenne de paires')
print('═'*70)
print(f'  Test .npy (officiel) : {acc_npy:.2%}')
print(f'  Test CSV (réel, FIXE, jamais augmenté) : {acc_csv:.2%}')
print('═'*70)
print()
print('  Comparaison avec le modèle mixte de référence (CSV NON augmenté, ~1864) :')
print(f'    Référence : Test npy=94.62%   Test CSV=99.25%')
print(f'    CE MODÈLE : Test npy={acc_npy:.2%}   Test CSV={acc_csv:.2%}')

In [ ]:
from torch.utils.tensorboard import SummaryWriter as SW2
HPARAMS_SUGGESTION2 = os.path.join(HOME, 'runs', 'hparams_sarun_suggestion2')

def log_run(cfg, metrics, run_label):
    run_dir = os.path.join(HPARAMS_SUGGESTION2, run_label)
    w = SW2(run_dir)
    hparams_clean = {k: v for k, v in cfg.items() if isinstance(v, (int, float, str, bool))}
    w.add_hparams(hparams_clean, metrics)
    w.close()

log_run(
    cfg={'csv_augmentation': 'pairwise_average', 'target_per_class': CFG['target_per_class'],
         'csv_train_n': len(df_csv_train_augmented)},
    metrics={'test_npy_acc': acc_npy, 'test_csv_acc': acc_csv},
    run_label='csv_augmented_18000',
)
print('✓ Run loggé')
print(f'Dashboard : tensorboard --logdir {HPARAMS_SUGGESTION2} --port 6014')